# ES Obfuscated Only

**Routing pattern:**
- WRITE: `[elasticsearch_obfuscated]` — only geoid tokens stored; no attributes persisted anywhere
- READ: `[elasticsearch_obfuscated]`
- SEARCH: `[elasticsearch_obfuscated]`
- METADATA: `[postgresql]` — collection metadata stays in PG registry

Use this for maximum anonymization where even the primary storage must not hold attributes.

In [ ]:
import httpx

BASE = "http://localhost:80"
CATALOG_ID = "demo-obfuscated-only"
COLLECTION_ID = "anon-locations"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

print("Client ready")

In [ ]:
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Obfuscated Only Demo", "description": "Obfuscated Only Demo catalog."})
_check(r)

In [ ]:
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID, "title": "Anonymous Locations",
    "description": "Anonymous Locations collection.",
    "license": "proprietary",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r)

In [ ]:
routing = {
    "operations": {
        "WRITE": [{"driver_id": "ItemsElasticsearchObfuscatedDriver", "on_failure": "fatal"}],
        "READ": [{"driver_id": "ItemsElasticsearchObfuscatedDriver"}],
        "SEARCH": [{"driver_id": "ItemsElasticsearchObfuscatedDriver"}],
        "METADATA": [{"driver_id": "ItemsPostgresqlDriver"}]
    }
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/classes/CollectionRoutingConfig",
    json=routing,
)
_check(r)

In [ ]:
features = [
    {
        "type": "Feature",
        "id": f"loc-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.01, 41.9 + i * 0.01]},
        "properties": {"pii_field": f"secret-{i}"}  # will NOT be stored
    }
    for i in range(5)
]
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json={"type": "FeatureCollection", "features": features},
)
_check(r, "5 items inserted (only geoids stored in ES)")

In [ ]:
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items?limit=10")
data = r.json()
print(f"Items: {len(data.get('features', []))}")
for f in data.get("features", []):
    # pii_field is absent — only geoid token returned
    assert "pii_field" not in f.get("properties", {}), "PII leaked!"
    print(" ", f.get("id"), "(no pii_field — correct)")